# Labeling Data Spasial per Wilayah dengan Random Forest

3 wilayah di Surabaya, 10 Juni 2026, dibedakan oleh gap waktu >1 jam:  
1. **Ngagel** (Pagi - Urban Traffic)  
2. **Rungkut** (Siang - Industri Manufaktur)  
3. **Margomulyo** (Sore - Industri Logistik)  

Model RF: `random_forest_air_quality.pkl` (sama dengan sistem prediksi 1 jam).  
Fitur: `PM2.5`, `PM10`, `CO`, `NO2`, `O3` → label ISPU.

In [16]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path.cwd().parent
DATA_SPASIAL = BASE / 'data_spasial'
MODEL_PATH = BASE / 'ml_model' / 'random_forest_air_quality.pkl'
RF_FEATURES = ['PM2.5', 'PM10', 'CO', 'NO2', 'O3']
COL_MAP = {
    'pm25_ugm3': 'PM2.5', 'pm10_ugm3': 'PM10', 'co_ugm3': 'CO',
    'no2_ugm3': 'NO2', 'o3_ugm3': 'O3'
}

In [17]:
rf_model = joblib.load(MODEL_PATH)
print(f'Model: {MODEL_PATH.name}')
print(f'Classes: {list(rf_model.classes_)}')

Model: random_forest_air_quality.pkl
Classes: ['Baik', 'Berbahaya', 'Sangat Tidak Sehat', 'Sedang', 'Tidak Sehat']


---
## 1. Load & Segmentasi Data Spasial

In [18]:
df = pd.read_csv(DATA_SPASIAL / 'tb_konsentrasi_gas_rows.csv', parse_dates=['created_at'])
print(f'Total data: {len(df)} baris')
print(f'Rentang: {df["created_at"].iloc[0]} - {df["created_at"].iloc[-1]}')

Total data: 190 baris
Rentang: 2026-06-09 23:39:21.128053+00:00 - 2026-06-10 09:49:39.106159+00:00


In [19]:
# Deteksi gap >1 jam (3600 detik)
time_diffs = df['created_at'].diff().dt.total_seconds()
gap_rows = time_diffs[time_diffs > 3600].index.tolist()

print(f'Gap >1 jam ditemukan: {len(gap_rows)}')
for idx in gap_rows:
    gap_min = time_diffs[idx] / 60
    print(f'  Baris {idx}: {gap_min:.0f} menit')
    print(f'    Sebelum: {df["created_at"].iloc[idx-1]}')
    print(f'    Sesudah: {df["created_at"].iloc[idx]}')

Gap >1 jam ditemukan: 3
  Baris 2: 86 menit
    Sebelum: 2026-06-09 23:40:22.227849+00:00
    Sesudah: 2026-06-10 01:05:54.483961+00:00
  Baris 62: 199 menit
    Sebelum: 2026-06-10 02:06:03.661717+00:00
    Sesudah: 2026-06-10 05:25:06.140920+00:00
  Baris 128: 120 menit
    Sebelum: 2026-06-10 06:47:30.719815+00:00
    Sesudah: 2026-06-10 08:47:27.396150+00:00


In [20]:
# Beri label segmen berdasarkan gap
seg = 0
segmen_labels = []
for i in range(len(df)):
    if i in gap_rows:
        seg += 1
    segmen_labels.append(seg)

df['segmen'] = segmen_labels
df['segmen'] = df['segmen'].replace(0, 1)  # gabung segmen 0 -> 1 (hanya 1 baris)

# Ambil semua data per segmen
segments = []
for s in sorted(df['segmen'].unique()):
    sub = df[df['segmen'] == s].copy()
    segments.append(sub)
df_seg = pd.concat(segments).sort_values('created_at').reset_index(drop=True)

LOKASI_MAP = {
    1: 'Ngagel (Pagi - Urban Traffic)',
    2: 'Rungkut (Siang - Industri Manufaktur)',
    3: 'Margomulyo (Sore - Industri Logistik)'
}
df_seg['lokasi'] = df_seg['segmen'].map(LOKASI_MAP)

for s in sorted(df_seg['segmen'].unique()):
    sub = df_seg[df_seg['segmen'] == s]
    print(f'{LOKASI_MAP[s]}: {len(sub)} baris, '
          f'{sub["created_at"].iloc[0]} - {sub["created_at"].iloc[-1]}')

Ngagel (Pagi - Urban Traffic): 62 baris, 2026-06-09 23:39:21.128053+00:00 - 2026-06-10 02:06:03.661717+00:00
Rungkut (Siang - Industri Manufaktur): 66 baris, 2026-06-10 05:25:06.140920+00:00 - 2026-06-10 06:47:30.719815+00:00
Margomulyo (Sore - Industri Logistik): 62 baris, 2026-06-10 08:47:27.396150+00:00 - 2026-06-10 09:49:39.106159+00:00


---
## 2. Label RF per Wilayah

In [21]:
X = df_seg[list(COL_MAP.keys())].rename(columns=COL_MAP)
df_seg['label_rf'] = rf_model.predict(X)

for s in sorted(df_seg['segmen'].unique()):
    sub = df_seg[df_seg['segmen'] == s]
    print(f'\n{"="*60}')
    print(f'Wilayah: {LOKASI_MAP[s]}')
    print(f'{"="*60}')
    print(sub[['created_at', 'pm25_ugm3', 'pm10_ugm3', 'co_ugm3', 'label_rf']].to_string(index=False))


Wilayah: Ngagel (Pagi - Urban Traffic)
                      created_at  pm25_ugm3  pm10_ugm3  co_ugm3    label_rf
2026-06-09 23:39:21.128053+00:00      28.11      30.01  1544.66      Sedang
2026-06-09 23:40:22.227849+00:00      29.87      31.87  1371.22      Sedang
2026-06-10 01:05:54.483961+00:00      25.11      31.41  1690.72      Sedang
2026-06-10 01:06:54.030364+00:00      28.64      32.57  1606.40      Sedang
2026-06-10 01:07:55.837918+00:00      26.17      33.50  1608.44      Sedang
2026-06-10 01:08:57.217019+00:00      26.17      42.32  1489.56      Sedang
2026-06-10 01:09:58.537479+00:00      35.33      36.52  1508.28 Tidak Sehat
2026-06-10 01:10:59.560795+00:00      30.22      36.52  1465.65      Sedang
2026-06-10 01:12:00.817427+00:00      26.87      30.94  1299.66      Sedang
2026-06-10 01:13:01.885353+00:00      24.93      30.71  1376.27      Sedang
2026-06-10 01:14:03.474985+00:00      31.10      33.03  1199.01      Sedang
2026-06-10 01:15:04.961537+00:00      24.93     

## 3. Ringkasan Distribusi Label per Wilayah (Sampled 60 per Wilayah)

In [22]:
# Load sampled dataset and show summary table with percentages
import pandas as pd
from pathlib import Path
BASE = Path.cwd().parent
sampled = pd.read_csv(BASE / 'analysis' / 'sampled_60_per_wilayah.csv', parse_dates=['created_at'])

summary = sampled.groupby(['lokasi', 'predicted_category']).size().reset_index(name='jumlah')
summary['persentase'] = summary.groupby('lokasi')['jumlah'].transform(lambda x: (x / x.sum() * 100).round(1).astype(str) + '%')
summary = summary.sort_values(['lokasi', 'predicted_category']).reset_index(drop=True)
print(summary.to_string(index=False))

# Save summary CSV
summary.to_csv(BASE / 'analysis' / 'sampled_summary_by_lokasi.csv', index=False)
print('\nWrote summary to sampled_summary_by_lokasi.csv')

    lokasi predicted_category  jumlah persentase
Margomulyo               Baik      53      88.3%
Margomulyo             Sedang       7      11.7%
    Ngagel             Sedang      57      95.0%
    Ngagel        Tidak Sehat       3       5.0%
   Rungkut               Baik       1       1.7%
   Rungkut             Sedang      59      98.3%

Wrote summary to sampled_summary_by_lokasi.csv
